# SNOMED CT — API & bulk

SNOMED CT is the canonical clinical terminology — diagnoses,
procedures, findings, body sites, etc. `biodb.snomed` covers both
modes:

| Mode | Functions |
|---|---|
| **API (OLS4 REST)** | `query_concept`, `search_concepts`, `get_descendants`, `get_ancestors`, `get_children`, `get_parents` |
| **Bulk (GitHub release)** | `download_concept_csv`, `load_concept_csv` — OHDSI-flavoured `CONCEPT.csv` |

The bulk file is the OMOP CDM concept dimension (~175 MB CSV, ~29 MB
compressed) — handy for joins against EHR data. Per-concept lookups
route through EBI's [OLS4](https://www.ebi.ac.uk/ols4/) (~376 k
SNOMED terms indexed) and accept the bare SNOMED ID
(`38341003`), the CURIE (`SNOMED:38341003`), or the full IRI
(`http://snomed.info/id/38341003`).

In [1]:
from biodb import snomed

## 1. API mode — per-concept lookups via OLS4

No local data required. Pass an `int`, a CURIE, or an IRI — they all
normalise to the same canonical record.

In [2]:
record = snomed.query_concept(38341003)
{
    "obo_id": record["obo_id"],
    "label": record["label"],
    "iri": record["iri"],
    "synonyms": record["synonyms"][:3] if record["synonyms"] else [],
    "is_obsolete": record["is_obsolete"],
}

{'obo_id': 'SNOMED:38341003',
 'label': 'Hypertensive disorder',
 'iri': 'http://snomed.info/id/38341003',
 'synonyms': [],
 'is_obsolete': False}

Full-text search across SNOMED concept labels / synonyms /
definitions. Solr-backed — fuzzy by default, pass `exact=True` to
require an exact match.

In [3]:
hits = snomed.search_concepts("hypertension", rows=5)
hits[["obo_id", "label"]]

,obo_id,label
0,SNOMED:10725009,Benign hypertension
1,SNOMED:123799005,Renovascular hypertension
2,SNOMED:123800009,Goldblatt hypertension
3,SNOMED:1356877007,Stable hypertension
4,SNOMED:162659009,Hypertension resolved


Walk the SNOMED IS-A graph — ancestors (up to the SCT root) and
descendants (the full subtree). Both return DataFrames; pagination
is handled internally.

In [4]:
parents = snomed.get_parents(38341003)
print(f"Hypertensive disorder has {len(parents)} direct parents:")
parents[["obo_id", "label"]]

Hypertensive disorder has 2 direct parents:


,obo_id,label
0,SNOMED:49601007,Disorder of cardiovascular system
1,SNOMED:366157005,Cardiovascular measurement - finding


In [5]:
descendants = snomed.get_descendants(38341003)
print(f"Hypertensive disorder has {len(descendants)} transitive descendants:")
descendants[["obo_id", "label"]].head(8)

Hypertensive disorder has 128 transitive descendants:


,obo_id,label
0,SNOMED:845891000000103,Hypertension resistant to drug therapy
1,SNOMED:84094009,Rebound hypertension
2,SNOMED:720568003,Brachydactyly and arterial hypertension syndrome
3,SNOMED:71701000119105,Hypertension in chronic kidney disease due to ...
4,SNOMED:712832005,Supine hypertension
5,SNOMED:71421000119105,Hypertension in chronic kidney disease due to ...
6,SNOMED:706882009,Hypertensive crisis
7,SNOMED:704667004,Hypertension concurrent and due to end stage r...


## 2. Bulk mode — OHDSI `CONCEPT.csv`

The release moved from `bschilder/synthlab` to `bschilder/bioDB` on
2026-05-18 — same bytes, identical SHA-256.

In [6]:
print(f"repo:    {snomed.GITHUB_REPO}")
print(f"release: {snomed.GITHUB_RELEASE_TAG}")
print(f"asset:   {snomed.GITHUB_ASSET_NAME}")
print(f"URL:     {snomed.SNOMED_RELEASE_URL}")

repo:    bschilder/bioDB
release: vocab-v1
asset:   CONCEPT.csv.gz
URL:     https://github.com/bschilder/bioDB/releases/download/vocab-v1/CONCEPT.csv.gz


### Authentication strategies (private mirrors only)

The bioDB release is public, so the third strategy below is always
what runs. The other two are there for downstream consumers who
fork bioDB into a private mirror.

1. **`gh` CLI** — if installed and authenticated
2. **`GITHUB_TOKEN` / `GH_TOKEN`** environment variable
3. **Public URL** (default)

In [7]:
print(f"gh CLI authenticated: {snomed._gh_cli_available()}")
print(f"env token set:        {snomed._get_github_token() is not None}")

gh CLI authenticated: True
env token set:        False


### Download + load

Smoke check — does the cached path exist already?

In [8]:
print(f"cached: {snomed.is_available()}")
print(f"cache dir: {snomed.CACHE_DIR}")

cached: False
cache dir: /Users/bschilder/.cache/biodb/snomed


Pull the asset (~29 MB compressed → ~175 MB CSV). Cached at
`~/.cache/biodb/snomed/CONCEPT.csv`; subsequent calls short-circuit.
Progress is reported via tqdm by default — pass `progress=False` to
silence.

```python
path = snomed.download_concept_csv()
```

`load_concept_csv` is the one-shot version — download (if needed)
plus `pd.read_csv` with the right dtype overrides.

```python
concept = snomed.load_concept_csv()
concept.shape  # → (~2.5 M rows, 10 columns)
concept[concept["vocabulary_id"] == "SNOMED"].head()
```

Both are skipped at notebook-execute time because the decompressed
CSV is 175 MB; running them locally is the right way to see them
work.

### Schema

The columns map 1-to-1 to OHDSI's CDM concept table — see
<https://ohdsi.github.io/CommonDataModel/cdm54.html#CONCEPT>.

In [9]:
expected_columns = [
    "concept_id",  # integer primary key
    "concept_name",
    "domain_id",  # Condition / Drug / Procedure / Measurement / Device / Observation / …
    "vocabulary_id",  # SNOMED, RxNorm, LOINC, …  (filter to "SNOMED" for the SNOMED slice)
    "concept_class_id",
    "standard_concept",  # 'S' = standard, 'C' = classification, NaN = non-standard
    "concept_code",  # the original SNOMED ID (e.g. 38341003 for hypertensive disorder)
    "valid_start_date",
    "valid_end_date",
    "invalid_reason",
]
for col in expected_columns:
    print(col)

concept_id
concept_name
domain_id
vocabulary_id
concept_class_id
standard_concept
concept_code
valid_start_date
valid_end_date
invalid_reason
